# Experiment Tracking — HiLiftAeroML point-cloud surrogate with W&B

The goal here is to fit a transolver++ model to predict pressure (Cp) on an airplane wing.

The data is demo-synthetic: slat / main-wing / flap-shaped point
clouds with a pressure-like Cp target. It stands in for the real
NASA CRM-HL boundary VTU files (13 GB each), so the pipeline runs
on a laptop CPU in a few minutes. The exact same W&B API calls
work when you swap in real HiLiftAeroML data.

**Prerequisites**
- **`registry.ipynb` executed first** — that notebook creates the
  `hilift` Registry and publishes the demo dataset at
  `wandb-registry-hilift/pointcloud_demo:production`, which this
  notebook consumes.
- We clone `thuml/Transolver_plus` from GitHub on
  first run.

**Reference materials**
- Blog: `https://hodgesj.substack.com/p/a-beginners-tutorial-the-new-hiliftaeroml`



## 1. Setup

Imports and section-local constants. Everything the training loop
and visualisation cells need is declared here.


In [ ]:
from __future__ import annotations

import importlib
import json
import math
import os
import subprocess
import sys
import tempfile
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import wandb

matplotlib.use("Agg")
plt.rcParams["figure.dpi"] = 110
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ---- W&B target ----
ENTITY = os.environ.get("WANDB_ENTITY")
PROJECT = os.environ.get("WANDB_PROJECT")

In [ ]:
REGISTRY_NAME = "hilift"
DATASET_COLLECTION = "pointcloud_demo"
CKPT_COLLECTION = "checkpoints"

# Training Config
CASE_SPECS = [
    ("LHC001",  4), ("LHC001",  8), ("LHC001", 12), ("LHC001", 16),
    ("LHC002",  4), ("LHC002",  8), ("LHC002", 12), ("LHC002", 16),
    ("LHC003",  4), ("LHC003",  8), ("LHC003", 12), ("LHC003", 16),
]

POINTS_PER_CASE_CACHE = 8192
PATCH_POINTS = 1024
PATCHES_PER_CASE = 96
BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-3

# Weights and Biases saving / logging cadence.
# Every N epochs we (a) log a checkpoint artifact aliased `epoch-N` and
# (b) log the 3D + 2D triptychs. §6 later resumes the run and iterates
# these `epoch-N` checkpoints to rebuild a full-test-set metrics table.
CHECKPOINT_PLOT_CADENCE = 10

TMP_DIR = Path(tempfile.mkdtemp(prefix="hilift_"))

print(f"torch  = {torch.__version__}")
print(f"wandb  = {wandb.__version__}")
print(f"device = {device}")
print(f"tmp    = {TMP_DIR}")


## 2. Transolver++ setup — clone + patch

We load the Transolver++ model from a github implementation [here](https://github.com/thuml/Transolver_plus.git).


`thuml/Transolver_plus` is the reference implementation of
Transolver++. It contains `dist_nn.all_reduce(...)` calls that
require a distributed process group to be initialised. We're running
single-process, so we patch those calls with a `_maybe_all_reduce`
helper that no-ops when no process group is active.

The repo is cached under `~/.cache/physicsx_demo/` so the
clone happens once.


In [ ]:
CACHE_ROOT = Path.home() / ".cache" / "physicsx_demo"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRANSOLVER_DIR = CACHE_ROOT / "Transolver_plus"

if not TRANSOLVER_DIR.exists():
    print(f"Cloning thuml/Transolver_plus into {TRANSOLVER_DIR} ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/thuml/Transolver_plus.git",
        str(TRANSOLVER_DIR),
    ])
else:
    print(f"Using cached Transolver_plus at {TRANSOLVER_DIR}")

model_file = TRANSOLVER_DIR / "models" / "Transolver_plus.py"
src_text = model_file.read_text()

if "def _maybe_all_reduce" not in src_text:
    src_text = src_text.replace(
        "import torch.distributed.nn as dist_nn\n",
        "import torch.distributed.nn as dist_nn\n"
        "import torch.distributed as dist\n\n\n"
        "def _maybe_all_reduce(tensor):\n"
        "    if dist.is_available() and dist.is_initialized() "
        "and dist.get_world_size() > 1:\n"
        "        dist_nn.all_reduce(tensor, op=dist_nn.ReduceOp.SUM)\n"
        "    return tensor\n",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_norm, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_norm)",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_token, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_token)",
    )
    model_file.write_text(src_text)
    print("Patched Transolver_plus.py with _maybe_all_reduce.")
else:
    print("Transolver_plus.py already patched.")

if str(TRANSOLVER_DIR) not in sys.path:
    sys.path.insert(0, str(TRANSOLVER_DIR))

if "models.Transolver_plus" in sys.modules:
    importlib.reload(sys.modules["models.Transolver_plus"])

from models.Transolver_plus import Model as TransolverPlusModel

print("Imported TransolverPlusModel OK.")


## 3. Prepare training tensors + model

Everything needed before the training loop starts. Broken into two
cells for readability:

1. Retrieve the dataset from the Registry, split 70 / 15 / 15
  case-level (with a point-level fallback for tiny case counts), and
  derive `point_mean/std` + `target_mean/std` on the *training subset
  only* so val/test stats stay honest.

2. The `HiLiftPointPatchDataset` (Transolver++ sees a fresh
  random `PATCH_POINTS`-sized patch each iteration), the
  `TransolverPointFieldSurrogate` wrapper around thuml's Model, and
  instantiation of both DataLoaders + `model` with a shape sanity
  check.


In [ ]:
def _load_npz_case(path: Path) -> dict:
    """Load one setup-notebook NPZ back into a plain-Python dict.

    Numeric arrays (points, normals, target) come back as numpy arrays.
    Scalar / string metadata (geo_id, run_id, target_name, ...) is
    unwrapped from its 0-D array container so downstream code sees
    plain Python types.
    """
    data = np.load(path, allow_pickle=True)
    case = {k: data[k] for k in data.files}

    for key in ["geo_id", "run_id", "target_name",
                "target_location", "target_mode"]:
        if key in case:
            case[key] = (
                str(case[key].item()) if case[key].ndim == 0 else str(case[key])
            )
    case["aoa"] = float(np.asarray(case["aoa"]).item())
    return case

def split_cases(
    cases_list: list[dict],
    ) -> tuple[list[dict], list[dict], list[dict], str]:
    
    """Split cases into train / val / test (70 / 15 / 15).

    With ≥ 3 cases we hold whole cases out — the more realistic
    generalisation test. With fewer, we shuffle points within each
    case and split point-level so the notebook still has train / val /
    test tensors to work with.

    Returns
    -------
    (train, val, test, split_mode) : 4-tuple
        `split_mode` is a short string used for reporting.
    """
    rng = np.random.default_rng(SEED)

    if len(cases_list) >= 3:
        order = np.arange(len(cases_list))
        rng.shuffle(order)

        n_train = max(1, int(0.70 * len(cases_list)))
        n_val = max(1, int(0.15 * len(cases_list)))

        train = [cases_list[i] for i in order[:n_train]]
        val = [cases_list[i] for i in order[n_train:n_train + n_val]]
        test = [cases_list[i] for i in order[n_train + n_val:]]

        if not test:                       # 3-case corner: reuse val for test
            test = val

        return train, val, test, "case split"

    # Point-level fallback for < 3 cases.
    train, val, test = [], [], []
    for case in cases_list:
        n = len(case["points"])
        order = np.arange(n)
        rng.shuffle(order)

        n_train = int(0.70 * n)
        n_val = int(0.15 * n)

        def _slice(idx: np.ndarray, suffix: str) -> dict:
            return {
                **case,
                "points": case["points"][idx],
                "normals": case["normals"][idx],
                "target": case["target"][idx],
                "run_id": f"{case['run_id']}_{suffix}",
            }

        train.append(_slice(order[:n_train], "train_points"))
        val.append(_slice(order[n_train:n_train + n_val], "val_points"))
        test.append(_slice(order[n_train + n_val:], "test_points"))

    return train, val, test, "point split within case(s)"

In [ ]:
# ==== Load + split + normalise =================================
# Consume the dataset from the Registry, split it into train/val/test,
# and derive per-channel normalisation stats on the training subset only.

# ---- Consume dataset from Registry ----
api = wandb.Api()
ds_art_info = api.artifact(
    f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
)

ds_dir = Path(ds_art_info.download())

cases = [_load_npz_case(p) for p in sorted(ds_dir.glob("*.npz"))]

print(f"loaded {len(cases)} cases from Registry:")
for c in cases:
    print(f"  {c['run_id']}   points={c['points'].shape}")


# ---- Case-level (or point-level fallback) train/val/test split ----
train_cases, val_cases, test_cases, split_mode = split_cases(cases)

TARGET_DIM = train_cases[0]["target"].shape[1]
print(f"\nSplit mode: {split_mode}")
print(f"  train: {[c['run_id'] for c in train_cases]}")
print(f"  val:   {[c['run_id'] for c in val_cases]}")
print(f"  test:  {[c['run_id'] for c in test_cases]}")


# ---- Normalisation stats (training subset only) ----
# One `stats` dict feeds every downstream consumer (DataLoader,
# predict_case, evaluate_patch_loader), so keeping the derivation in
# one place is important.
train_points_all = np.concatenate([c["points"] for c in train_cases], axis=0)
train_target_all = np.concatenate([c["target"] for c in train_cases], axis=0)

stats = {
    "point_mean":  train_points_all.mean(axis=0).astype(np.float32),
    "point_std":   np.maximum(train_points_all.std(axis=0), 1e-6).astype(np.float32),
    "target_mean": train_target_all.mean(axis=0).astype(np.float32),
    "target_std":  np.maximum(train_target_all.std(axis=0), 1e-6).astype(np.float32),
    "aoa_mean":    np.float32(np.mean([c["aoa"] for c in train_cases])),
    "aoa_std":     np.float32(max(np.std([c["aoa"] for c in train_cases]), 1.0)),
}

print(f"\ntarget mean/std: {stats['target_mean']} / {stats['target_std']}")

In [ ]:
# ==== PyTorch Dataset + Transolver++ wrapper + instantiation ===
# All the class + helper definitions the training loop depends on,
# followed by concrete objects (`train_loader`, `val_loader`, `model`).

# ---- Feature builder: encode AoA as a 3-D conditioning vector ----
def aoa_condition(aoa_degrees: float, stats: dict) -> np.ndarray:
    """Encode angle of attack as `(z-norm, sin(rad), cos(rad))`.

    A single float doesn't give the model enough affordance to learn
    periodic behaviour; passing three redundant encodings — a
    standard-normalised scalar plus sinusoidal features — helps the
    conditioning stay smooth across AoA.
    """
    aoa_norm = (np.float32(aoa_degrees) - stats["aoa_mean"]) / stats["aoa_std"]
    aoa_rad = np.deg2rad(np.float32(aoa_degrees))
    return np.array(
        [aoa_norm, np.sin(aoa_rad), np.cos(aoa_rad)],
        dtype=np.float32,
    )


# ---- PyTorch Dataset: random point-patches per case ----
class HiLiftPointPatchDataset(Dataset):
    """Yield random `patch_points`-sized subsets of each case per epoch.

    Transolver++ sees a fresh random patch every iteration. Over an
    epoch's `patches_per_case` iterations the model implicitly covers
    the full point cloud without ever seeing all N points at once.
    """

    def __init__(
        self,
        case_list: list[dict],
        stats: dict,
        patch_points: int,
        patches_per_case: int,
        random_patches: bool = True,
    ) -> None:
        self.case_list = case_list
        self.stats = stats
        self.patch_points = patch_points
        self.patches_per_case = patches_per_case
        self.random_patches = random_patches
        self.rng = np.random.default_rng(SEED)

    def __len__(self) -> int:
        return len(self.case_list) * self.patches_per_case

    def __getitem__(
        self, idx: int,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        case = self.case_list[idx % len(self.case_list)]
        n = len(case["points"])
        replace = n < self.patch_points

        # Random patch during training; deterministic per-index during val
        # so metrics don't jitter epoch-to-epoch.
        if self.random_patches:
            point_idx = self.rng.choice(n, size=self.patch_points, replace=replace)
        else:
            rng = np.random.default_rng(SEED + idx)
            point_idx = rng.choice(n, size=self.patch_points, replace=replace)

        points = case["points"][point_idx].astype(np.float32)
        normals = case["normals"][point_idx].astype(np.float32)
        target = case["target"][point_idx].astype(np.float32)

        # Standard-normalise inputs and targets using train-only stats.
        pos = (points - self.stats["point_mean"]) / self.stats["point_std"]
        y = (target - self.stats["target_mean"]) / self.stats["target_std"]

        # Per-point features: [normalised position (3), surface normal (3)]
        x = np.concatenate([pos, normals], axis=1).astype(np.float32)
        cond = aoa_condition(case["aoa"], self.stats)

        return (
            torch.tensor(x),
            torch.tensor(pos.astype(np.float32)),
            torch.tensor(cond),
            torch.tensor(y.astype(np.float32)),
        )


# ---- Model wrapper around thuml's Transolver++ ----
class TransolverPointFieldSurrogate(nn.Module):
    """Point-cloud → per-point scalar prediction via Transolver++.

    Wraps the reference `TransolverPlusModel` and adapts its
    `(x, pos, condition)` triple-input signature into a single
    `forward` method the rest of the notebook calls uniformly.
    """

    def __init__(
        self,
        feature_dim: int = 6,
        out_dim: int = 1,
        n_hidden: int = 96,
        n_layers: int = 4,
        n_head: int = 4,
        slice_num: int = 16,
        dropout: float = 0.05,
    ) -> None:
        super().__init__()
        self.backbone = TransolverPlusModel(
            space_dim=feature_dim,
            fun_dim=0,
            out_dim=out_dim,
            n_hidden=n_hidden,
            n_layers=n_layers,
            n_head=n_head,
            mlp_ratio=2,
            slice_num=slice_num,
            dropout=dropout,
            unified_pos=False,
        )

    def forward(
        self,
        x: torch.Tensor,
        pos: torch.Tensor,
        condition: torch.Tensor,
    ) -> torch.Tensor:
        return self.backbone((x, pos, condition))



In [ ]:
# ---- Instantiate DataLoaders + model, sanity-check one forward pass ----
train_loader = DataLoader(
    HiLiftPointPatchDataset(
        train_cases, stats, PATCH_POINTS, PATCHES_PER_CASE,
        random_patches=True,
    ),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
)
val_loader = DataLoader(
    HiLiftPointPatchDataset(
        val_cases, stats, PATCH_POINTS,
        max(16, PATCHES_PER_CASE // 4),
        random_patches=False,
    ),
    batch_size=BATCH_SIZE, shuffle=False,
)

TARGET_DIM = train_cases[0]["target"].shape[1]

model = TransolverPointFieldSurrogate(feature_dim=6, out_dim=TARGET_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Transolver++ point-field surrogate: {n_params:,} trainable params")

# One forward pass to confirm shapes align before the training run starts.
xb, posb, cb, yb = next(iter(train_loader))
print(
    f"batch shapes: x={tuple(xb.shape)} pos={tuple(posb.shape)} "
    f"cond={tuple(cb.shape)} y={tuple(yb.shape)}"
)

## 4. Train + log to W&B + promote best checkpoint

One `wandb.init(job_type="train")` block covers the entire training.

- **Training helpers: normalisation + evaluation.** The
  denormalisation tensors plus `evaluate_patch_loader` and
  `predict_case`. These are the "how do we score the model"
  primitives the training loop calls each epoch and the §5 close-up
  reuses.
- **Training helpers: metrics + visualisation.**
  `regression_metrics` (per-component MAE / RMSE / R²) and the two
  triptych plot functions — 3D rotatable Plotly + 2D matplotlib
  top-view — that get logged to W&B as media panels.


- **Main training loop + promotion.** The `with
  wandb.init(...) as run:` block itself:
  - Consumes the dataset for lineage
  - per epoch logs `train/loss` + `val_metrics/*` +
  `learning_rate`
  - every `CHECKPOINT_PLOT_CADENCE=10` epochs (and
  whenever val improves) saves a checkpoint artifact aliased
  `epoch-N` — §6 resumes onto these
  - every cadence epoch logs 3D + 2D triptychs
  - Promotes the best-val
  version to `hilift/checkpoints:production`.
  - Captures `TRAIN_RUN_ID = run.id` so §6 can resume the same run.


In [ ]:
# ==== Training helpers: normalisation + evaluation ==============
# Denormalisation tensors + `evaluate_patch_loader` + `predict_case`.
# These are the "how do we score the model" primitives the training
# loop and close-up both call.

# ---- Denormalisation tensors on the training device ----
# Both `evaluate_patch_loader` and `predict_case` convert normalised
# outputs back to physical Cp so they can report MAE / RMSE in the
# same units engineers use. Cache the tensors once here so we don't
# rebuild them per batch.
target_mean_t: torch.Tensor = torch.tensor(
    stats["target_mean"], dtype=torch.float32, device=device,
)
target_std_t: torch.Tensor = torch.tensor(
    stats["target_std"], dtype=torch.float32, device=device,
)

def denorm_target(y_norm: torch.Tensor) -> torch.Tensor:
    """Undo the standard-normalisation applied by `HiLiftPointPatchDataset`.

    Model outputs live in normalised (unit-std, zero-mean) space. This
    helper maps them back to physical target units.
    """
    return y_norm * target_std_t + target_mean_t


@torch.no_grad()
def evaluate_patch_loader(
    loader: DataLoader,
) -> tuple[float, float, float]:
    """Run the model over `loader` in eval mode.

    Returns
    -------
    (loss, mae, rmse) : tuple of floats
        - `loss` — mean normalised MSE (matches the training objective).
        - `mae`  — mean absolute error on denormalised predictions,
          physical target units.
        - `rmse` — RMSE on denormalised predictions, physical target units.
    """
    model.eval()
    torch.manual_seed(SEED)                # deterministic patch sampling

    losses: list[float] = []
    maes: list[float] = []
    rmses: list[float] = []

    for x, pos, cond, y in loader:
        x = x.to(device); pos = pos.to(device)
        cond = cond.to(device); y = y.to(device)

        pred = model(x, pos, cond)
        loss = F.mse_loss(pred, y)

        pred_phys = denorm_target(pred)
        y_phys = denorm_target(y)
        err = pred_phys - y_phys

        losses.append(loss.item())
        maes.append(err.abs().mean().item())
        rmses.append(torch.sqrt((err ** 2).mean()).item())

    return float(np.mean(losses)), float(np.mean(maes)), float(np.mean(rmses))


@torch.no_grad()
def predict_case(case: dict, chunk_points: int = 4096) -> np.ndarray:
    """Score a full case's ~8k points in `chunk_points`-sized batches.

    CPU can't run all N points through Transolver++ at once, so we
    chunk. Predictions come back denormalised, shape `(N, out_dim)`.

    Parameters
    ----------
    case : dict
        A case dict as produced by `_load_npz_case` — must contain
        `points (N, 3)`, `normals (N, 3)`, `target (N, D)`, `aoa`.
    chunk_points : int
        Chunk size for the streaming forward pass.
    """
    model.eval()
    torch.manual_seed(SEED)

    points = case["points"].astype(np.float32)
    normals = case["normals"].astype(np.float32)

    # Precompute normalised positions + concat with normals once.
    pos_all = ((points - stats["point_mean"]) / stats["point_std"]).astype(np.float32)
    x_all = np.concatenate([pos_all, normals], axis=1).astype(np.float32)

    cond = torch.tensor(
        aoa_condition(case["aoa"], stats),
        dtype=torch.float32, device=device,
    ).unsqueeze(0)

    preds: list[np.ndarray] = []
    for start in range(0, len(points), chunk_points):
        end = min(start + chunk_points, len(points))

        x = torch.tensor(x_all[start:end], dtype=torch.float32, device=device).unsqueeze(0)
        pos = torch.tensor(pos_all[start:end], dtype=torch.float32, device=device).unsqueeze(0)

        pred_norm = model(x, pos, cond)
        preds.append(denorm_target(pred_norm).squeeze(0).cpu().numpy())

    return np.concatenate(preds, axis=0)

In [ ]:
# ==== Training helpers: metrics + visualisation =================
# Per-component regression metrics + the two triptych plot functions
# the training loop logs to W&B as media panels.

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """Per-component MAE / RMSE / R² as a pandas DataFrame.

    Component-wise reporting means multi-target models (e.g. predicting
    Cp *and* wall-shear jointly) still get a clean summary row per
    channel. For a single-target run the DataFrame has one row.
    """
    rows: list[dict] = []
    for j in range(y_true.shape[1]):
        rows.append({
            "component": j,
            "MAE":  float(mean_absolute_error(y_true[:, j], y_pred[:, j])),
            "RMSE": float(np.sqrt(mean_squared_error(y_true[:, j], y_pred[:, j]))),
            "R2":   float(r2_score(y_true[:, j], y_pred[:, j])),
        })
    return pd.DataFrame(rows)


def plot_prediction_triptych(
    case: dict,
    truth: np.ndarray,
    pred: np.ndarray,
    max_points: int = 25_000,
) -> go.Figure:
    """3D rotatable Plotly triptych: truth / prediction / abs error.

    Uses `aspectmode="cube"` so each scene fills its 1/3 of the panel
    regardless of the point cloud's underlying anisotropy. Downsamples
    to `max_points` for interactivity — Plotly gets sluggish beyond
    ~50k markers.
    """
    points = case["points"]
    truth_scalar = truth[:, 0]
    pred_scalar = pred[:, 0]
    err_scalar = np.abs(pred_scalar - truth_scalar)

    # Downsample for browser interactivity.
    rng = np.random.default_rng(SEED)
    if len(points) > max_points:
        idx = rng.choice(len(points), size=max_points, replace=False)
        points = points[idx]
        truth_scalar = truth_scalar[idx]
        pred_scalar = pred_scalar[idx]
        err_scalar = err_scalar[idx]

    # Shared colour scale for truth + prediction so the eye can compare.
    cmin = float(min(truth_scalar.min(), pred_scalar.min()))
    cmax = float(max(truth_scalar.max(), pred_scalar.max()))

    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
        subplot_titles=("CFD truth", "Transolver++ prediction", "absolute error"),
    )
    panels = [
        (truth_scalar, "Viridis", cmin, cmax, "truth"),
        (pred_scalar,  "Viridis", cmin, cmax, "pred"),
        (err_scalar,   "Magma",   0.0, float(np.percentile(err_scalar, 99)), "abs error"),
    ]

    for col, (scalar, cs, lo, hi, name) in enumerate(panels, start=1):
        # Each scene occupies x-domain [(col-1)/3, col/3]; place its
        # colorbar at x = col/3 - 0.02 so it sits snug against the scene.
        cbar_x = col / 3.0 - 0.02
        fig.add_trace(
            go.Scatter3d(
                x=points[:, 0], y=points[:, 1], z=points[:, 2],
                mode="markers",
                marker=dict(
                    size=2, color=scalar, colorscale=cs,
                    cmin=lo, cmax=hi, opacity=0.82,
                    colorbar=dict(
                        title=name, x=cbar_x, y=0.5,
                        len=0.7, thickness=10,
                    ),
                ),
                name=name, showlegend=False,
            ),
            row=1, col=col,
        )

    fig.update_layout(
        title=f"{case['run_id']} | target: {case['target_name']}",
        height=520,
        margin=dict(l=0, r=0, t=60, b=0),
    )
    # aspectmode="cube" — each scene renders as a unit cube regardless of
    # data extents. Without this the y-extent (~6.4) dominates x (~2.5)
    # and z (~0.7) and each scene collapses into a thin band.
    fig.update_scenes(
        aspectmode="cube",
        xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
    )
    return fig


def plot_prediction_2d_triptych(
    case: dict,
    truth: np.ndarray,
    pred: np.ndarray,
    max_points: int = 30_000,
) -> plt.Figure:
    """2D top-down triptych: truth / prediction / abs error.

    Static PNG counterpart to the 3D Plotly view. Span (y) goes on the
    horizontal axis because span is the wider data dimension and matches
    the wider W&B panel shape; chord (x) goes vertical.
    """
    points = case["points"]
    truth_scalar = truth[:, 0]
    pred_scalar = pred[:, 0]
    err_scalar = np.abs(pred_scalar - truth_scalar)

    # Downsample for figure-render speed.
    rng = np.random.default_rng(SEED)
    idx = np.arange(len(points))
    if len(idx) > max_points:
        idx = rng.choice(idx, size=max_points, replace=False)

    # figsize picked so `set_aspect("equal")` fills each subplot without
    # leaving thin strips — data span ~6.4 (y) x ~2.5 (x) → wide+short.
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), constrained_layout=True)

    cmin = min(truth_scalar.min(), pred_scalar.min())
    cmax = max(truth_scalar.max(), pred_scalar.max())
    plots = [
        (truth_scalar, "CFD truth",  "viridis", cmin, cmax),
        (pred_scalar,  "prediction", "viridis", cmin, cmax),
        (err_scalar,   "abs error",  "magma",
         0.0, float(np.percentile(err_scalar, 99))),
    ]

    for ax, (values, title, cmap, lo, hi) in zip(axes, plots):
        # Swap: y (span) on horizontal, x (chord) on vertical.
        sc = ax.scatter(
            points[idx, 1], points[idx, 0],
            c=values[idx], s=2, cmap=cmap, vmin=lo, vmax=hi,
        )
        ax.set_aspect("equal", adjustable="box")
        ax.set_title(title)
        ax.set_xlabel("y (span)")
        ax.set_ylabel("x (chord)")
        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle(f"{case['run_id']} — 2D top view", y=1.02)
    return fig


In [ ]:
training_config = {
        "epochs": EPOCHS, 
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "patch_points": PATCH_POINTS,
        "patches_per_case": PATCHES_PER_CASE,
        "points_per_case_cache": POINTS_PER_CASE_CACHE,
        "n_train_cases": len(train_cases),
        "n_val_cases": len(val_cases),
        "n_test_cases": len(test_cases),
        "n_params": n_params,
        "model": {
            "feature_dim": 6, "out_dim": TARGET_DIM,
            "n_hidden": 96, "n_layers": 4, "n_head": 4,
            "slice_num": 16, "dropout": 0.05, "mlp_ratio": 2,
        },
        "case_specs": [list(cs) for cs in CASE_SPECS],
        "split_mode": split_mode,
    }

In [ ]:
# ==== Main training loop + promotion ============================
# One `with wandb.init(...) as run:` block covers training, per-epoch
# logging, checkpoint promotion, and dataset lineage. At the end we
# link the best-val checkpoint into `hilift/checkpoints` on the
# Registry so any downstream consumer (Sweeps, Reports, other teams)
# can pull the final model by canonical path.
#
# We capture the run id into `TRAIN_RUN_ID` at the end of the block —
# §6 uses it to resume this same run and append a full-test-set
# metrics table.

with wandb.init(
    entity=ENTITY, project=PROJECT,
    job_type="train",
    name="hilift-transolverpp-train",
    tags=["hilift", "transolver++"],
    config=training_config,
) as run:

    run.use_artifact(
        f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
    )

    # ---- Ensure the model Registry exists (idempotent) ----
    api_train = wandb.Api()
    try:
        api_train.create_registry(name=REGISTRY_NAME, visibility="organization")
        print(f"created registry: {REGISTRY_NAME}")
    except Exception as exc:
        print(f"{REGISTRY_NAME} create skipped: {exc}")

    # ---- Optimiser + LR schedule ----
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    # ---- Training-loop state ----
    best_val: float = float("inf")
    best_ckpt_artifact = None
    best_epoch: int | None = None
    global_step: int = 0

    # ============ Epoch loop ============
    for epoch in range(1, EPOCHS + 1):

        # -- Training pass --
        model.train()
        train_losses: list[float] = []
        
        for x, pos, cond, y in train_loader:
            x = x.to(device)
            pos = pos.to(device)
            cond = cond.to(device)
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x, pos, cond)
            loss = F.mse_loss(pred, y)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_losses.append(loss.item())
            global_step += 1
        
        scheduler.step()

        # -- Validation + per-epoch metric log --
        train_loss_epoch = float(np.mean(train_losses))
        val_loss, val_mae, val_rmse = evaluate_patch_loader(val_loader)
        rmse_norm = val_rmse / float(stats["target_std"].mean())

        # Log training and validation metrics to WB.
        wandb.log(
            {
            "epoch": epoch,
            "train/loss": train_loss_epoch,
            "val_metrics/loss": val_loss,
            "val_metrics/mae": val_mae,
            "val_metrics/rmse": val_rmse,
            "val_metrics/rmse_norm": rmse_norm,
            "learning_rate": float(scheduler.get_last_lr()[0]),
            }, 
            step=global_step)

        # -- Checkpoint every N epochs OR whenever we beat best-val --
        # The "always log the best" carve-out guarantees the best-val
        # version is a genuine artifact and can be linked to the Registry.
        # The `epoch-N` aliases are what §6 resumes onto — do not remove.
        is_best = val_loss < best_val
        if epoch % CHECKPOINT_PLOT_CADENCE == 0 or is_best:
            ckpt_path = TMP_DIR / f"hilift_ckpt_ep{epoch:03d}.pt"
            
            torch.save({
                "state_dict": model.state_dict(),
                "stats": stats,
                "epoch": epoch,
                "val_loss": val_loss,
                "val_mae": val_mae,
                "val_rmse": val_rmse,
                "target_dim": TARGET_DIM,
            }, ckpt_path)

            ckpt_art = wandb.Artifact(
                name="hilift_transolverpp_ckpt",
                type="model",
                metadata={
                    "epoch": epoch,
                    "val_loss": val_loss,
                    "val_mae": val_mae,
                    "val_rmse": val_rmse,
                    "n_params": n_params,
                },
            )
            ckpt_art.add_file(str(ckpt_path), name="checkpoint.pt")
            logged = run.log_artifact(ckpt_art, aliases=[f"epoch-{epoch}"])

            if is_best:
                best_val = val_loss
                best_epoch = epoch
                best_ckpt_artifact = logged

        # -- Media panels every N epochs --
        if epoch % CHECKPOINT_PLOT_CADENCE == 0:
            vis_case = test_cases[0]
            pred_vis = predict_case(vis_case)
            fig_3d = plot_prediction_triptych(vis_case, vis_case["target"], pred_vis)
            fig_2d = plot_prediction_2d_triptych(vis_case, vis_case["target"], pred_vis)
            wandb.log({
                f"val_plots/predictions_epoch_{epoch}": wandb.Plotly(fig_3d),
                f"val_plots/predictions_2d_epoch_{epoch}": wandb.Image(fig_2d),
            }, step=global_step)
            
            plt.close(fig_2d)

        # -- Console progress print --
        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
            print(
                f"epoch {epoch:03d} | train {train_loss_epoch:.4f} | "
                f"val {val_loss:.4f} | mae {val_mae:.4g} | rmse {val_rmse:.4g}"
            )

    # ============ Promote best checkpoint into the Registry ============
    best_ckpt_artifact.wait()
    target_path = f"wandb-registry-{REGISTRY_NAME}/{CKPT_COLLECTION}"
    best_alias = f"best_epoch_{best_epoch}"
    
    run.link_artifact(
        best_ckpt_artifact,
        target_path=target_path,
        aliases=["production", best_alias, "best"],
    )

    run.summary.update({
        "best_epoch": best_epoch,
        "best_val_loss": best_val,
    })

    # Capture the run id so §6 can resume this exact run.
    TRAIN_RUN_ID = run.id

    print()
    print(f"best epoch: {best_epoch}   val_loss={best_val:.4f}")
    print(f"linked to {target_path}   aliases: production, {best_alias}")
    print(f"train run id: {TRAIN_RUN_ID}")


## 5. Inference on held-out cases

Outside the training run, reload the best checkpoint straight from
the Registry — the same path any downstream consumer (a Report, a
Sweep, another team) would use. Then run `predict_case` on every
test case and print regression metrics + render the 3D triptychs
inline.


In [ ]:
def load_best_from_registry() -> dict:
    """Download and load the `production`-aliased checkpoint.

    Returns the raw checkpoint dict as saved by the training cell
    (`state_dict`, `stats`, `epoch`, `val_loss`, ...).
    """
    api = wandb.Api()
    best_art = api.artifact(
        f"wandb-registry-{REGISTRY_NAME}/{CKPT_COLLECTION}:production"
    )
    best_dir = Path(best_art.download())
    return torch.load(
        next(best_dir.glob("*.pt")),
        map_location=device,
        weights_only=False,
    )

In [32]:
# ==== Post-training prediction close-up ==========================
# Reload the best-val checkpoint from the Registry (same path any
# downstream consumer would use) and inspect predictions on every test
# case inline.

ckpt = load_best_from_registry()

print(f"\nLoaded best checkpoint from Registry: epoch {ckpt['epoch']}\n\n")
model.load_state_dict(ckpt["state_dict"])
model.eval()

# Iterate every test case, print per-component metrics, and render the rotatable 3D triptych inline for visual inspection.
for case in test_cases[0:1]:
    pred = predict_case(case)
    m = regression_metrics(case["target"], pred)

    print(f"{case['run_id']}   AoA={case['aoa']}°")
    print("="*40)
    print(m.round(5).to_string(index=False))

    fig = plot_prediction_triptych(case, case["target"], pred)
    fig.show()
    print()


wandb:   1 of 1 files downloaded.  



Loaded best checkpoint from Registry: epoch 50


demo_geo_LHC001_AoA_8   AoA=8.0°
 component     MAE    RMSE      R2
         0 0.01208 0.01517 0.99805


## 6. Resume the training run and write a full-test-set metrics table

The training loop only logs *scalar* val metrics per epoch. To attach
a richer, per-case view of test performance across the training
trajectory, we **resume the same run** and append a `wandb.Table`
under a new key: `test_table/regression_metrics`.

**Why resume, not a new run?** Keeping the analysis attached to the
training run means one URL holds everything a reviewer needs: config,
loss curves, media panels, and per-case test metrics.

**Why is this safe here?** Other notebooks in this workshop
(`5_advanced_multistage_training_pipeline`,
`6-advanced_asynchronous_evals`) explicitly warn against
`wandb.init(resume=...)` because concurrent resumers cause config
overwrites, step interleaving, `run.finish()` races, and last-write-
wins on summary. None of those apply here:

- **Sequential**: training's `run.finish()` fired before this cell
  opens the run — no concurrency.
- **Fresh key**: we log under `test_table/regression_metrics`, which
  no other writer touches — no last-write-wins.
- **No `config` mutation** and no `wandb.log(step=...)` with a
  colliding step counter.
- **`resume="must"`** fails loudly if `TRAIN_RUN_ID` is stale rather
  than silently forking a new run.

**How we rebuild the table:** each cadence epoch (`10, 20, 30, 40,
50`) has a checkpoint artifact aliased `epoch-N`. We iterate them,
load the weights, and score **every** held-out test case (unlike the
in-training loop, this is not sampled). Rows: `len(cadence_epochs) ×
len(test_cases)`.

> **Re-execution note.** Re-running this cell appends another table
> version under the same key — W&B shows the most recent version, so
> the panel displays the last write. Prior versions remain in the
> artifact history but aren't rendered.


In [ ]:
# ==== Resume the training run + append full-test-set table ======
# Sequential resume: `resume="must"` errors loudly if TRAIN_RUN_ID is
# wrong, rather than silently starting a new run. See the markdown
# above for why this specific pattern is safe.

with wandb.init(
    entity=ENTITY, project=PROJECT,
    id=TRAIN_RUN_ID,
    resume="must",
) as run:
    api_resume = wandb.Api()

    # test_table = wandb.Table(
    #     columns=["epoch", "run_id", "aoa", "MAE", "RMSE", "R2"],
    # )

    cadence_epochs = list(range(
        CHECKPOINT_PLOT_CADENCE, EPOCHS + 1, CHECKPOINT_PLOT_CADENCE,
    ))

    table_rows = []
    for epoch in cadence_epochs:
        # load the checkpoint for this epoch from the Registry, not local disk, so we can resume on any machine without needing the original TMP_DIR.
        ckpt_art = api_resume.artifact(
            f"{ENTITY}/{PROJECT}/hilift_transolverpp_ckpt:epoch-{epoch}"
        )
        ckpt_dir = Path(ckpt_art.download())
        ckpt = torch.load(
            next(ckpt_dir.glob("*.pt")),
            map_location=device, weights_only=False,
        )
        model.load_state_dict(ckpt["state_dict"])
        model.eval()

        # Predict for each instance in test case
        for case in test_cases:
            pred = predict_case(case)
            r = regression_metrics(case["target"], pred)
            table_row = {
                "epoch": epoch,
                "run_id": case["run_id"],
                "aoa": case["aoa"],
                "MAE": float(r["MAE"].mean()),
                "RMSE": float(r["RMSE"].mean()),
                "R2": float(r["R2"].mean()),
            }
            table_rows.append(table_row)
            
    test_table = wandb.Table(dataframe=pd.DataFrame(table_rows))
    run.log({"test_predictions/regression_metrics": test_table})

    n_rows = len(cadence_epochs) * len(test_cases)
    print(f"appended {n_rows} rows to resumed run {TRAIN_RUN_ID}")
    print(f"  cadence epochs: {cadence_epochs}")
    print(f"  test cases:     {[c['run_id'] for c in test_cases]}")
